## StockTopic — Bronze → Silver → Gold Pipeline

Parses the Kafka-ingested `finance.bronze.stocktopic` (3 event types) into structured silver and gold tables.

| Layer | Table | Source Event | Description |
|-------|-------|-------------|-------------|
| Silver | `stock_ticks` | TICK | Real-time price ticks (symbol, price, bid, ask, volume) |
| Silver | `stock_ohlcv` | OHLCV | End-of-day OHLCV bars with VWAP |
| Silver | `portfolio_holdings_cdc` | CDC | Portfolio change-data-capture events |
| Gold | `realtime_tick_summary` | TICK | Latest & avg price, spread, total volume per symbol |
| Gold | `daily_ohlcv_enriched` | OHLCV | Daily bars with return %, volatility, rolling avg |
| Gold | `portfolio_latest_holdings` | CDC | Materialized latest portfolio state per user/symbol |

In [0]:
# =========================
# CONFIG & HELPERS
# =========================
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, get_json_object, from_json, to_timestamp,
    sum as _sum, count, avg, min as _min, max as _max,
    round as _round, when, lit, row_number, lag, abs as _abs
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    LongType, IntegerType, TimestampType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BRONZE = "finance.bronze"
SILVER = "finance.silver"
GOLD = "finance.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

def upsert(df_source, schema, table_name, merge_keys):
    """Incremental MERGE: INSERT new, UPDATE existing."""
    full_table = f"{schema}.{table_name}"
    df_source = df_source.withColumn("_loaded_at", F.current_timestamp())
    if spark.catalog.tableExists(full_table):
        dt = DeltaTable.forName(spark, full_table)
        cond = " AND ".join([f"t.{k} = s.{k}" for k in merge_keys])
        dt.alias("t").merge(
            df_source.alias("s"), cond
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        action = "MERGED"
    else:
        df_source.write.format("delta").saveAsTable(full_table)
        action = "CREATED"
    cnt = spark.table(full_table).count()
    print(f"  \u2705 {full_table} \u2014 {cnt:,} rows ({action})")

print("\u2705 Config ready")

In [0]:
# =========================
# READ BRONZE SOURCE & SPLIT BY EVENT TYPE
# =========================
df_raw = spark.table(f"{BRONZE}.stocktopic")

# Extract event_type from JSON value
df_raw = df_raw.withColumn("event_type", get_json_object("value", "$.event_type"))

df_ticks_raw = df_raw.filter(col("event_type") == "TICK")
df_ohlcv_raw = df_raw.filter(col("event_type") == "OHLCV")
df_cdc_raw   = df_raw.filter(col("event_type") == "CDC")

print(f"Bronze total:  {df_raw.count():,}")
print(f"  TICK:  {df_ticks_raw.count():,}")
print(f"  OHLCV: {df_ohlcv_raw.count():,}")
print(f"  CDC:   {df_cdc_raw.count():,}")

In [0]:
# =========================
# SILVER 1: STOCK TICKS
# Parse TICK events into structured columns
# =========================
print("\U0001f539 SILVER: stock_ticks")

df_ticks = (
    df_ticks_raw
    .select(
        get_json_object("value", "$.sequence_id").cast("long").alias("sequence_id"),
        to_timestamp(get_json_object("value", "$.timestamp")).alias("tick_timestamp"),
        get_json_object("value", "$.symbol").alias("symbol"),
        get_json_object("value", "$.price").cast("double").alias("price"),
        get_json_object("value", "$.bid").cast("double").alias("bid"),
        get_json_object("value", "$.ask").cast("double").alias("ask"),
        get_json_object("value", "$.volume").cast("long").alias("volume"),
        get_json_object("value", "$.exchange").alias("exchange"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    )
    .withColumn("spread", _round(col("ask") - col("bid"), 4))
    .withColumn("spread_pct", _round((col("ask") - col("bid")) / col("price") * 100, 4))
)

upsert(df_ticks, SILVER, "stock_ticks", ["sequence_id"])

In [0]:
# =========================
# SILVER 2: STOCK OHLCV
# Parse OHLCV end-of-day bars
# =========================
print("\U0001f539 SILVER: stock_ohlcv")

df_ohlcv = (
    df_ohlcv_raw
    .select(
        get_json_object("value", "$.sequence_id").cast("long").alias("sequence_id"),
        to_timestamp(get_json_object("value", "$.timestamp")).alias("event_timestamp"),
        get_json_object("value", "$.date").cast("date").alias("trade_date"),
        get_json_object("value", "$.symbol").alias("symbol"),
        get_json_object("value", "$.open").cast("double").alias("open"),
        get_json_object("value", "$.high").cast("double").alias("high"),
        get_json_object("value", "$.low").cast("double").alias("low"),
        get_json_object("value", "$.close").cast("double").alias("close"),
        get_json_object("value", "$.volume").cast("long").alias("volume"),
        get_json_object("value", "$.vwap").cast("double").alias("vwap"),
        get_json_object("value", "$.trades_count").cast("long").alias("trades_count"),
        get_json_object("value", "$.source").alias("source"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    )
    .withColumn("daily_range", _round(col("high") - col("low"), 4))
    .withColumn("daily_range_pct", _round((col("high") - col("low")) / col("open") * 100, 2))
)

upsert(df_ohlcv, SILVER, "stock_ohlcv", ["sequence_id"])

In [0]:
# =========================
# SILVER 3: PORTFOLIO HOLDINGS CDC
# Parse CDC change-data-capture events
# =========================
print("\U0001f539 SILVER: portfolio_holdings_cdc")

df_cdc = (
    df_cdc_raw
    .select(
        get_json_object("value", "$.sequence_id").cast("long").alias("sequence_id"),
        to_timestamp(get_json_object("value", "$.timestamp")).alias("event_timestamp"),
        get_json_object("value", "$.db_table").alias("db_table"),
        get_json_object("value", "$.operation").alias("operation"),
        get_json_object("value", "$.transaction_id").alias("transaction_id"),
        # Before state
        get_json_object("value", "$.before.user_id").alias("before_user_id"),
        get_json_object("value", "$.before.symbol").alias("before_symbol"),
        get_json_object("value", "$.before.quantity").cast("int").alias("before_quantity"),
        get_json_object("value", "$.before.status").alias("before_status"),
        # After state
        get_json_object("value", "$.after.user_id").alias("after_user_id"),
        get_json_object("value", "$.after.symbol").alias("after_symbol"),
        get_json_object("value", "$.after.quantity").cast("int").alias("after_quantity"),
        get_json_object("value", "$.after.avg_cost").cast("double").alias("after_avg_cost"),
        get_json_object("value", "$.after.status").alias("after_status"),
        # Source metadata
        get_json_object("value", "$.source.lsn").alias("source_lsn"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    )
    # Coalesce symbol from before/after (UPDATEs have it in before, INSERTs in after)
    .withColumn("symbol", F.coalesce(col("after_symbol"), col("before_symbol")))
    .withColumn("user_id", F.coalesce(col("after_user_id"), col("before_user_id")))
    .withColumn("qty_change", 
        when(col("operation") == "INSERT", col("after_quantity"))
        .when(col("operation") == "DELETE", -col("before_quantity"))
        .otherwise(col("after_quantity") - col("before_quantity"))  # UPDATE
    )
)

upsert(df_cdc, SILVER, "portfolio_holdings_cdc", ["sequence_id"])

In [0]:
# =========================
# GOLD 1: REALTIME TICK SUMMARY
# Latest & avg price, spread, total volume per symbol
# =========================
print("\U0001f4b0 GOLD: realtime_tick_summary")

df_ticks_silver = spark.table(f"{SILVER}.stock_ticks")

# Latest tick per symbol
w_latest = Window.partitionBy("symbol").orderBy(F.desc("sequence_id"))

df_latest = (
    df_ticks_silver
    .withColumn("rn", row_number().over(w_latest))
    .filter(col("rn") == 1)
    .select(
        col("symbol"),
        col("price").alias("latest_price"),
        col("bid").alias("latest_bid"),
        col("ask").alias("latest_ask"),
        col("spread").alias("latest_spread"),
        col("exchange"),
        col("tick_timestamp").alias("last_tick_time")
    )
)

# Aggregated stats per symbol
df_agg = (
    df_ticks_silver
    .groupBy("symbol")
    .agg(
        _round(avg("price"), 4).alias("avg_price"),
        _round(_min("price"), 4).alias("min_price"),
        _round(_max("price"), 4).alias("max_price"),
        _round(avg("spread"), 4).alias("avg_spread"),
        _round(avg("spread_pct"), 4).alias("avg_spread_pct"),
        _sum("volume").alias("total_volume"),
        count("*").alias("tick_count")
    )
)

df_tick_summary = (
    df_latest.join(df_agg, "symbol", "left")
    .withColumn("price_range", _round(col("max_price") - col("min_price"), 4))
    .withColumn("price_range_pct", _round(
        when(col("min_price") > 0,
             (col("max_price") - col("min_price")) / col("min_price") * 100
        ).otherwise(0), 2))
)

upsert(df_tick_summary, GOLD, "realtime_tick_summary", ["symbol"])

In [0]:
# =========================
# GOLD 2: DAILY OHLCV ENRICHED
# Daily bars with return %, volatility metrics
# =========================
print("\U0001f4c8 GOLD: daily_ohlcv_enriched")

df_ohlcv_silver = spark.table(f"{SILVER}.stock_ohlcv")

# Use the latest OHLCV per symbol per date (in case of duplicates)
w_dedup = Window.partitionBy("symbol", "trade_date").orderBy(F.desc("sequence_id"))

df_ohlcv_dedup = (
    df_ohlcv_silver
    .withColumn("rn", row_number().over(w_dedup))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Add daily return and prev close using window
w_daily = Window.partitionBy("symbol").orderBy("trade_date")

df_daily = (
    df_ohlcv_dedup
    .withColumn("prev_close", lag("close").over(w_daily))
    .withColumn("daily_return", _round(
        when(col("prev_close").isNotNull() & (col("prev_close") > 0),
             (col("close") - col("prev_close")) / col("prev_close") * 100
        ).otherwise(0), 2))
    .withColumn("gap_pct", _round(
        when(col("prev_close").isNotNull() & (col("prev_close") > 0),
             (col("open") - col("prev_close")) / col("prev_close") * 100
        ).otherwise(0), 2))
    .withColumn("body_pct", _round(_abs(col("close") - col("open")) / col("open") * 100, 2))
    .withColumn("is_bullish", when(col("close") >= col("open"), True).otherwise(False))
    .withColumn("ohlcv_key", F.concat_ws("_", "symbol", F.col("trade_date").cast("string")))
    .select(
        "ohlcv_key", "symbol", "trade_date", "open", "high", "low", "close",
        "volume", "vwap", "trades_count", "daily_range", "daily_range_pct",
        "prev_close", "daily_return", "gap_pct", "body_pct", "is_bullish", "source"
    )
)

upsert(df_daily, GOLD, "daily_ohlcv_enriched", ["ohlcv_key"])

In [0]:
# =========================
# GOLD 3: PORTFOLIO LATEST HOLDINGS
# Materialized current state from CDC events
# =========================
print("\U0001f464 GOLD: portfolio_latest_holdings")

df_cdc_silver = spark.table(f"{SILVER}.portfolio_holdings_cdc")

# Latest CDC event per user + symbol (most recent state)
w_latest_cdc = Window.partitionBy("user_id", "symbol").orderBy(F.desc("sequence_id"))

df_latest_holdings = (
    df_cdc_silver
    .withColumn("rn", row_number().over(w_latest_cdc))
    .filter(col("rn") == 1)
    .filter(col("operation") != "DELETE")  # Exclude deleted positions
)

# Per-user/symbol aggregated CDC stats
df_cdc_stats = (
    df_cdc_silver
    .groupBy("user_id", "symbol")
    .agg(
        count("*").alias("total_cdc_events"),
        count(when(col("operation") == "INSERT", True)).alias("insert_count"),
        count(when(col("operation") == "UPDATE", True)).alias("update_count"),
        count(when(col("operation") == "DELETE", True)).alias("delete_count"),
        _sum("qty_change").alias("net_qty_change"),
        _min("event_timestamp").alias("first_event_time"),
        _max("event_timestamp").alias("last_event_time")
    )
)

df_holdings = (
    df_latest_holdings
    .join(df_cdc_stats, ["user_id", "symbol"], "left")
    .withColumn("holding_key", F.concat_ws("_", "user_id", "symbol"))
    .select(
        "holding_key",
        "user_id", "symbol",
        col("after_quantity").alias("current_quantity"),
        col("after_avg_cost").alias("avg_cost"),
        col("after_status").alias("status"),
        col("operation").alias("last_operation"),
        col("transaction_id").alias("last_txn_id"),
        "total_cdc_events", "insert_count", "update_count", "delete_count",
        "net_qty_change", "first_event_time", "last_event_time"
    )
)

upsert(df_holdings, GOLD, "portfolio_latest_holdings", ["holding_key"])

In [0]:
# =========================
# PIPELINE SUMMARY
# =========================
tables = {
    "SILVER": ["stock_ticks", "stock_ohlcv", "portfolio_holdings_cdc"],
    "GOLD": ["realtime_tick_summary", "daily_ohlcv_enriched", "portfolio_latest_holdings"]
}

print("=" * 60)
print("\U0001f4ca STOCKTOPIC PIPELINE SUMMARY")
print("=" * 60)
for layer, tbls in tables.items():
    schema = SILVER if layer == "SILVER" else GOLD
    print(f"\n  {layer}:")
    for tbl in tbls:
        full = f"{schema}.{tbl}"
        if spark.catalog.tableExists(full):
            cnt = spark.table(full).count()
            cols = len(spark.table(full).columns)
            print(f"    \u2705 {full:<48} {cnt:>6,} rows  |  {cols} cols")
        else:
            print(f"    \u23f3 {full:<48} not yet created")
print("\n" + "=" * 60)
print("\u2705 StockTopic pipeline complete!")